# no-grad-context-mgr-update — ex1: implement NoGrad context manager with restore-on-exit

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `no-grad-context-mgr-update`. Running the final beacon cell reports progress against the `Backprop: no_grad ctx-mgr update` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: no_grad ctx-mgr update` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`no-grad-context-mgr-update`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "no-grad-context-mgr-update"
DD_SUBTOPIC = "Backprop: no_grad ctx-mgr update"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `no_grad` context manager — quick refresher

A context manager that flips the module-level `grad_tracking_enabled` flag to `False` for the duration of the `with` block, and restores the previous value on exit:

```python
class NoGrad:
    def __enter__(self):
        global grad_tracking_enabled
        self._prev = grad_tracking_enabled
        grad_tracking_enabled = False
    def __exit__(self, *exc):
        global grad_tracking_enabled
        grad_tracking_enabled = self._prev
```

Three things matter:
- **Save the previous value** (don't just set `True` on exit) so nesting   works — an inner `NoGrad` that exits doesn't accidentally re-enable   grad inside an outer `NoGrad`.
- **Restore on exit even if the block raises** — `__exit__` is   guaranteed to fire; that's the contract.
- **Use cases:** in-place parameter updates inside an optimizer step,   EMA buffers, inference paths — anywhere you DON'T want a Recipe   attached to the output.

### Exercise 1 — implement NoGrad context manager with restore-on-exit

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the save-and-restore context-manager pattern to write NoGrad: flip grad_tracking_enabled to False on enter, restore the previous value on exit (so nesting works).
> Keywords: context-manager, no-grad, grad-tracking, restore
> ```

**KCs targeted:** `no-grad-context-mgr-update`, `grad-tracking-global-toggle`

Implement `NoGrad` — a context manager that disables the module-level `grad_tracking_enabled` flag for the duration of the `with` block, then restores the **previous** value on exit:

```python
with NoGrad():
    # grad_tracking_enabled is False here
    ...
# grad_tracking_enabled is back to whatever it was before
```

Two requirements:

**1. Save the PREVIOUS value, not just `True`.** Hard-coding `grad_tracking_enabled = True` on exit breaks nested `with NoGrad()` blocks: the inner exit would re-enable grad inside the outer `NoGrad`. Read the current value in `__enter__`, stash it, restore it in `__exit__`.

**2. Restore on exit even if the block raised.** `__exit__` always fires (that's the contract), so the simplest correct implementation already gets this for free. Just don't `return` out of `__exit__` early on the exception path.

`grad_tracking_enabled` is a **module-level** name — use `global` to write to it.

Don't touch `torch.autograd`; we're rebuilding the autograd layer by hand.

In [ ]:
class NoGrad:
    """Disable grad_tracking_enabled for the duration of a `with` block."""
    def __enter__(self):
        raise NotImplementedError()
    def __exit__(self, exc_type, exc_val, exc_tb):
        raise NotImplementedError()


def _test_ex1():
    # --- baseline: grad_tracking_enabled starts True (from the preamble) ---
    assert grad_tracking_enabled is True, 'preamble must seed True'

    # --- inside the `with`, the flag flips to False ---
    with NoGrad():
        assert grad_tracking_enabled is False, (
            'NoGrad must disable grad_tracking_enabled on enter'
        )

    # --- after exit, the flag is restored ---
    assert grad_tracking_enabled is True, (
        'NoGrad must restore grad_tracking_enabled on exit'
    )

    # --- restore the PREVIOUS value, not hard-coded True ---
    # (Simulate caller having already disabled grad themselves. We write to the
    #  same module dict that NoGrad's `global` will read/write.)
    globals()['grad_tracking_enabled'] = False
    with NoGrad():
        assert grad_tracking_enabled is False
    assert grad_tracking_enabled is False, (
        'NoGrad should restore the PRE-EXISTING value (False), not hard-code True'
    )
    globals()['grad_tracking_enabled'] = True  # reset before next check

    # --- nesting: inner NoGrad exit must NOT re-enable grad ---
    with NoGrad():
        assert grad_tracking_enabled is False, 'outer NoGrad disabled'
        with NoGrad():
            assert grad_tracking_enabled is False, 'inner NoGrad still disabled'
        # back inside the OUTER NoGrad — must still be False
        assert grad_tracking_enabled is False, (
            'inner NoGrad exit must restore previous value (False), '
            'NOT hard-code True'
        )
    assert grad_tracking_enabled is True, 'outer exit restores original True'

    # --- restore on exit even when the block raises ---
    try:
        with NoGrad():
            assert grad_tracking_enabled is False
            raise RuntimeError('boom')
    except RuntimeError:
        pass
    assert grad_tracking_enabled is True, (
        'NoGrad must restore the flag even if the block raises an exception'
    )

    # --- __enter__ returns the manager (or None) — both are acceptable ---
    with NoGrad() as ng:
        pass  # just exercise the protocol, don't constrain the return value

    # --- usage in a parameter-update setting (the canonical use case) ---
    # Inside NoGrad, an 'update' should not flip the flag back.
    param_array = t.tensor([1.0, 2.0])
    lr = 0.1
    grad = t.tensor([0.5, -0.2])
    with NoGrad():
        # canonical optimizer.step()-style in-place update
        param_array -= lr * grad
        assert grad_tracking_enabled is False, (
            'grad must stay disabled during the in-place update'
        )
    assert grad_tracking_enabled is True
    assert t.allclose(param_array, t.tensor([0.95, 2.02])), (
        f'in-place update inside NoGrad must still mutate the tensor: {param_array}'
    )
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
class NoGrad:
    def __enter__(self):
        global grad_tracking_enabled
        self._prev = grad_tracking_enabled   # stash so nesting works
        grad_tracking_enabled = False
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        global grad_tracking_enabled
        grad_tracking_enabled = self._prev   # restore PREVIOUS value
        # Return None / False so any in-block exception is re-raised.
```

**Why save `self._prev` instead of toggling True on exit.** Three scenarios all need the saved-previous-value behavior:
1. Caller had already disabled grad (e.g. they're running inference and called `set_grad_enabled(False)` once at top of main). A naive `__exit__` setting True would silently re-enable.
2. Nested `with NoGrad(): with NoGrad(): ...` — inner exit must leave the outer block disabled.
3. Cross-test isolation — pytest fixtures often want the flag preserved across context exits.

All three reduce to 'save what you saw on entry; restore it.'

**Why no try/finally needed.** `__exit__` is called by the interpreter as part of the `with` protocol regardless of whether the body raised; we don't need to wrap anything. Returning `None` (implicit) from `__exit__` tells Python NOT to suppress exceptions — which is what we want.

**Equivalent with `contextlib.contextmanager`.** A `@contextmanager` generator + try/yield/finally version would also work, but the class form is what ARENA uses and what PyTorch's `torch.no_grad()` looks like internally.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()